In [1]:
# %pip install --user imbalanced-learn

In [6]:
#PRUEBA COMPARATIVA ENTRE ONE-HOT Y K-MERS CON RandomForestClassifier

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import requests
import time

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score


base = pd.read_csv("/home/jbs1009/TFM/datosGene4PD/base_flanqueantes_20_OR_3109_con_rsid_extendida.csv")
etiqueta_binaria = {"Sano": 0, "Riesgo_PD": 1}
base["labels"] = base["labels"].map(etiqueta_binaria)


X_seq = base["Secuencia"]
y = base["labels"].values
grupos_rsid = base["rsID"].values

encoding = {"a": [1, 0, 0, 0], "c": [0, 1, 0, 0], "g": [0, 0, 1, 0], "t": [0, 0, 0, 1]}

def one_hot_encode_sequence(seq):
    return np.array([encoding.get(nuc, [0, 0, 0, 0]) for nuc in seq])

X_oh = np.array([one_hot_encode_sequence(seq) for seq in X_seq])
X_oh = X_oh.reshape(X_oh.shape[0], -1)


def kmeriza(sequence, k=3):
    return [sequence[i:i+k] for i in range(len(sequence) - k + 1)]

base['kmers'] = X_seq.apply(lambda seq: ' '.join(kmeriza(seq, k=3)))

gss = GroupShuffleSplit(n_splits = 1, test_size = 0.25, random_state = 2026)
train_id, test_id = next(gss.split(X_seq, y, groups = grupos_rsid))

y_train, y_test = y[train_id], y[test_id]

X_train_oh, X_test_oh = X_oh[train_id], X_oh[test_id]

vectorizer = CountVectorizer()
X_train_kmers = vectorizer.fit_transform(base["kmers"].iloc[train_id]).toarray()
X_test_kmers = vectorizer.transform(base["kmers"].iloc[test_id]).toarray()


model_oh = RandomForestClassifier(n_estimators = 200, class_weight = "balanced", random_state = 2026, n_jobs = -1)
model_kmers = RandomForestClassifier(n_estimators = 200, class_weight = "balanced", random_state = 2026, n_jobs = -1)

model_oh.fit(X_train_oh, y_train)
model_kmers.fit(X_train_kmers, y_train)

y_pred_proba_oh = model_oh.predict_proba(X_test_oh)[:, 1]
y_pred_proba_kmers = model_kmers.predict_proba(X_test_kmers)[:, 1]

auc_roc_oh = roc_auc_score(y_test, y_pred_proba_oh)
auc_roc_kmers = roc_auc_score(y_test, y_pred_proba_kmers)

auc_pr_oh = average_precision_score(y_test, y_pred_proba_oh)
auc_pr_kmers = average_precision_score(y_test, y_pred_proba_kmers)

print(f'ROC-AUC One-Hot: {auc_roc_oh:.4f}')
print(f'ROC-AUC k-mers: {auc_roc_kmers:.4f}')

print(f'PR-AUC One-Hot: {auc_pr_oh:.4f}')
print(f'PR-AUC k-mers: {auc_pr_kmers:.4f}')

print("Classification Report (One-Hot)\n")
print(classification_report(y_test, model_oh.predict(X_test_oh)))

print("Classification Report (K-mers)\n")
print(classification_report(y_test, model_kmers.predict(X_test_kmers)))

ROC-AUC One-Hot: 0.5697
ROC-AUC k-mers: 0.6109
PR-AUC One-Hot: 0.2623
PR-AUC k-mers: 0.2882
Classification Report (One-Hot)

              precision    recall  f1-score   support

           0       0.78      1.00      0.88       599
           1       0.00      0.00      0.00       171

    accuracy                           0.78       770
   macro avg       0.39      0.50      0.44       770
weighted avg       0.61      0.78      0.68       770

Classification Report (K-mers)

              precision    recall  f1-score   support

           0       0.78      1.00      0.88       599
           1       0.00      0.00      0.00       171

    accuracy                           0.78       770
   macro avg       0.39      0.50      0.44       770
weighted avg       0.61      0.78      0.68       770



/home/jbs1009/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/jbs1009/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/jbs1009/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/jbs1009/.local/lib/python3.8/si

In [1]:
#PRUEBA COMPARATIVA ENTRE ONE-HOT Y K-MERS CON MLPClassifier

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import requests
import time

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score


base = pd.read_csv("/home/jbs1009/TFM/datosGene4PD/base_flanqueantes_20_OR_3109_con_rsid_extendida.csv")
etiqueta_binaria = {"Sano": 0, "Riesgo_PD": 1}
base["labels"] = base["labels"].map(etiqueta_binaria)


X_seq = base["Secuencia"]
y = base["labels"].values
grupos_rsid = base["rsID"].values

encoding = {"a": [1, 0, 0, 0], "c": [0, 1, 0, 0], "g": [0, 0, 1, 0], "t": [0, 0, 0, 1]}

def one_hot_encode_sequence(seq):
    return np.array([encoding.get(nuc, [0, 0, 0, 0]) for nuc in seq])

X_oh = np.array([one_hot_encode_sequence(seq) for seq in X_seq])
X_oh = X_oh.reshape(X_oh.shape[0], -1)


def kmeriza(sequence, k=3):
    return [sequence[i:i+k] for i in range(len(sequence) - k + 1)]

base['kmers'] = X_seq.apply(lambda seq: ' '.join(kmeriza(seq, k=3)))

gss = GroupShuffleSplit(n_splits = 1, test_size = 0.25, random_state = 2026)
train_id, test_id = next(gss.split(X_seq, y, groups = grupos_rsid))

y_train, y_test = y[train_id], y[test_id]

X_train_oh, X_test_oh = X_oh[train_id], X_oh[test_id]

vectorizer = CountVectorizer()
X_train_kmers = vectorizer.fit_transform(base["kmers"].iloc[train_id]).toarray()
X_test_kmers = vectorizer.transform(base["kmers"].iloc[test_id]).toarray()

scaler = StandardScaler()
X_train_kmers = scaler.fit_transform(X_train_kmers)
X_test_kmers = scaler.transform(X_test_kmers)



model_oh = MLPClassifier(hidden_layer_sizes = (20,), max_iter = 500, random_state = 2026)
model_kmers = MLPClassifier(hidden_layer_sizes = (20,), max_iter = 500, random_state = 2026)

model_oh.fit(X_train_oh, y_train)
model_kmers.fit(X_train_kmers, y_train)

y_pred_proba_oh = model_oh.predict_proba(X_test_oh)[:, 1]
y_pred_proba_kmers = model_kmers.predict_proba(X_test_kmers)[:, 1]

auc_roc_oh = roc_auc_score(y_test, y_pred_proba_oh)
auc_roc_kmers = roc_auc_score(y_test, y_pred_proba_kmers)

auc_pr_oh = average_precision_score(y_test, y_pred_proba_oh)
auc_pr_kmers = average_precision_score(y_test, y_pred_proba_kmers)

print(f'ROC-AUC One-Hot: {auc_roc_oh:.4f}')
print(f'ROC-AUC k-mers: {auc_roc_kmers:.4f}')

print(f'PR-AUC One-Hot: {auc_pr_oh:.4f}')
print(f'PR-AUC k-mers: {auc_pr_kmers:.4f}')

print("Classification Report (One-Hot)\n")
print(classification_report(y_test, model_oh.predict(X_test_oh)))

print("Classification Report (K-mers)\n")
print(classification_report(y_test, model_kmers.predict(X_test_kmers)))

/home/jbs1009/.local/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


ROC-AUC One-Hot: 0.5280
ROC-AUC k-mers: 0.5923
PR-AUC One-Hot: 0.2323
PR-AUC k-mers: 0.2963
Classification Report (One-Hot)

              precision    recall  f1-score   support

           0       0.78      0.86      0.82       599
           1       0.23      0.15      0.18       171

    accuracy                           0.70       770
   macro avg       0.51      0.50      0.50       770
weighted avg       0.66      0.70      0.68       770

Classification Report (K-mers)

              precision    recall  f1-score   support

           0       0.79      0.87      0.83       599
           1       0.31      0.19      0.24       171

    accuracy                           0.72       770
   macro avg       0.55      0.53      0.53       770
weighted avg       0.68      0.72      0.70       770



/home/jbs1009/.local/lib/python3.8/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
